# Graph Attention Networks (GAT & GATv2) on Cora

Node Classification on Cora (Planetoid): Multi-head attention mechanisms assigning dynamic importance weights to graph edges. This notebook implements the approach with `GATConv / GATv2Conv` inside a `K3GAT` model, trained with the Adam optimizer for 100 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GATConv / GATv2Conv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Planetoid
from k3_node import transforms as k3_transforms

title = "Graph Attention Networks (GAT & GATv2) on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora", transform=k3_transforms.NormalizeFeatures())
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. GAT Model Definition
class K3GAT(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super().__init__()
        self.conv1 = k3_layers.GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        self.conv2 = k3_layers.GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)
        self.dropout = layers.Dropout(0.6)

    def call(self, inputs, edge_index=None, training=False):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = self.dropout(x, training=training)
        x = ops.elu(self.conv1(x, edge_index))
        x = self.dropout(x, training=training)
        return self.conv2(x, edge_index)

k3_model = K3GAT(num_features, 8, num_classes, heads=8)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005, weight_decay=5e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# 4. Generator & Training
def graph_data_generator():
    x = ops.convert_to_tensor(data.x, dtype="float32")
    edge_index = ops.convert_to_tensor(data.edge_index, dtype="int64")
    y = ops.convert_to_tensor(data.y, dtype="int64")
    mask = ops.cast(data.train_mask, "float32")
    while True:
        yield (x, edge_index), y, mask

print(f"Training K3-Node GAT on {backend} backend...")
history = k3_model.fit(
    graph_data_generator(),
    steps_per_epoch=1,
    epochs=20,
    verbose=1,
)

# 5. Evaluation
out = k3_model((data.x, data.edge_index))
pred = ops.argmax(out, axis=-1)
test_mask = data.test_mask
test_acc = ops.mean(ops.cast(ops.cast(pred[test_mask], "int64") == ops.cast(data.y[test_mask], "int64"), "float32"))
print(f"Test Accuracy: {float(test_acc):.4f}")

print("\n✓ K3-Node execution completed successfully!")
